In [1]:
!pip install -q "transformers>=4.51" peft accelerate pandas matplotlib
!pip uninstall -y torchao

In [2]:
# ===================== CELL 0: setup =====================
import os, re, json, math, time, random, glob
from collections import Counter, defaultdict
from functools import lru_cache
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as Fn
from PIL import Image
from tqdm.auto import tqdm

SMOKE  = False                        # True = tiny 20-update dry run of all 3 stages; set False for the real run
SEED   = 6304
N_VIS  = 196                         # 49 for the optional compression run
R_COCO = "/kaggle/input/datasets/nadaibrahim/coco2014"
R_VQA  = "/kaggle/input/datasets/subhankashif/vqa-v2/v2_Complementary_Pairs_Val_mscoco"
WORK   = "/kaggle/working"
OUT    = f"{WORK}/run_v{N_VIS}" + ("_smoke" if SMOKE else ""); os.makedirs(OUT, exist_ok=True)
FEATS_NPY, ROWS_JSON, DATA_JSON = f"{WORK}/feats.npy", f"{WORK}/feat_rows.json", f"{WORK}/data_split.json"
QWEN, SIGLIP = "Qwen/Qwen3-0.6B", "google/siglip-base-patch16-224"
MIN_AREA = 1600
CAPQ = "Describe this image briefly."
dev = "cuda"

def find(name):                       # COCO files (captions/instances/images are nested at unknown depth)
    r = glob.glob(f"{R_COCO}/**/{name}", recursive=True); assert r, f"not found: {name}"; return r[0]
def vq(name):                         # VQA files (all 6 sit in one folder)
    p = f"{R_VQA}/{name}"; assert os.path.exists(p), f"not found: {p}"; return p
IMG_DIR = {"train": os.path.dirname(find("COCO_train2014_000000000009.jpg")),
           "val":   os.path.dirname(find("COCO_val2014_000000000042.jpg"))}
print(IMG_DIR)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

{'train': '/kaggle/input/datasets/nadaibrahim/coco2014/train2014/train2014', 'val': '/kaggle/input/datasets/nadaibrahim/coco2014/val2014/val2014'}


In [3]:
# ===================== CELL 1: VQA answer normalisation + scoring =====================
PUNCT = [';', r"/", '[', ']', '"', '{', '}', '(', ')', '=', '+', '\\', '_', '-', '>', '<', '@', '`', ',', '?', '!']
PERIOD, COMMA = re.compile(r"(?!<=\d)(\.)(?!\d)"), re.compile(r"(\d)(,)(\d)")
NUM = {"none":"0","zero":"0","one":"1","two":"2","three":"3","four":"4","five":"5","six":"6","seven":"7","eight":"8","nine":"9","ten":"10"}
ART = {"a","an","the"}
CONTR = {"dont":"don't","doesnt":"doesn't","isnt":"isn't","arent":"aren't","cant":"can't","couldnt":"couldn't","didnt":"didn't",
         "hasnt":"hasn't","havent":"haven't","im":"i'm","ive":"i've","itll":"it'll","shouldnt":"shouldn't","wasnt":"wasn't",
         "werent":"weren't","wont":"won't","wouldnt":"wouldn't","thats":"that's","theres":"there's","whats":"what's"}
def normalize_answer(s):
    s = s.replace("\n"," ").replace("\t"," ").strip().lower(); out = s
    for p in PUNCT:
        if (p+" " in s or " "+p in s) or COMMA.search(s): out = out.replace(p, "")
        else: out = out.replace(p, " ")
    out = PERIOD.sub("", out)
    w = [NUM.get(x, x) for x in out.split()]
    w = [x for x in w if x not in ART]
    return " ".join(CONTR.get(x, x) for x in w)

def vqa_soft(pred_norm, refs):
    a = [normalize_answer(r) for r in refs]
    return float(np.mean([min(1.0, sum(pred_norm == a[j] for j in range(len(a)) if j != i) / 3) for i in range(len(a))]))

In [4]:
# ===================== CELL 2: data construction (seed 6304) =====================
PLUR = {"person":"people","bus":"buses","knife":"knives","bench":"benches","couch":"couches","sandwich":"sandwiches",
        "wine glass":"wine glasses","mouse":"mice","sheep":"sheep","skis":"skis","scissors":"scissors","broccoli":"broccoli"}
plural = lambda n: PLUR.get(n, n + "s")
flat = lambda pairs: [e for p in pairs for e in p["ex"]]

def mk_ex(img, q, a, kind, typ, **kw):
    return dict(img=img, q=q, a=a, kind=kind, type=typ, canon=normalize_answer(a), refs=kw.pop("refs", [a]), **kw)
def mk_pair(pid, typ, exs):
    exs[0]["partner_img"], exs[1]["partner_img"] = exs[1]["img"], exs[0]["img"]
    return dict(pid=pid, type=typ, ex=exs)
def tag(pairs, name):
    for k, p in enumerate(pairs):
        p["pid"] = f"{name}-{k}"
        for e in p["ex"]: e["pid"] = p["pid"]
    return pairs
def assign_mismatch(pairs, seed):
    r = random.Random(seed)
    for pi, p in enumerate(pairs):
        for e in p["ex"]:
            qj = r.choice([j for j in range(len(pairs)) if j != pi]); e["mm_img"] = r.choice(pairs[qj]["ex"])["img"]

def load_caps(sp):
    m = defaultdict(list)
    for a in json.load(open(find(f"captions_{sp}2014.json")))["annotations"]: m[a["image_id"]].append(a["caption"].strip())
    return m
def load_inst(sp):
    d = json.load(open(find(f"instances_{sp}2014.json")))
    cats = {c["id"]: c["name"] for c in d["categories"]}
    raw, clean = defaultdict(Counter), defaultdict(Counter)
    for a in d["annotations"]:
        i, c = a["image_id"], a["category_id"]; raw[i][c] += 1
        if a["iscrowd"] == 0 and a["area"] >= MIN_AREA: clean[i][c] += 1
    return cats, raw, clean

def make_ground(cats, raw, clean, images, quota, rng, used=None):
    images = list(images); pos = defaultdict(list); byk = defaultdict(lambda: defaultdict(list))
    for i in images:
        for c, n in clean[i].items():
            if n >= 1: pos[c].append(i)
            if n <= 4 and raw[i][c] == n: byk[c][n].append(i)
    ok = lambda i: used is None or i not in used
    out = []
    for kind, n in quota.items():
        pc = sorted(pos) if kind == "presence" else sorted(c for c in byk if sum(bool(v) for v in byk[c].values()) >= 2)
        got = tries = 0
        while got < n:
            tries += 1; assert tries < 300000, "could not fill quota"
            c = rng.choice(pc); name = cats[c]
            if kind == "presence":
                a, b = rng.choice(pos[c]), rng.choice(images)
                if a == b or raw[b][c] != 0 or not (ok(a) and ok(b)): continue
                q, ans = f"Does this image contain {name}?", ("yes", "no")
            else:
                ks = [k for k in byk[c] if byk[c][k]]; k1, k2 = rng.sample(ks, 2)
                a, b = rng.choice(byk[c][k1]), rng.choice(byk[c][k2])
                if a == b or not (ok(a) and ok(b)): continue
                q, ans = f"How many {plural(name)} can you see?", (str(k1), str(k2))
            if used is not None: used.update([a, b])
            exs = [mk_ex(a, q, ans[0], kind, kind, cat=name), mk_ex(b, q, ans[1], kind, kind, cat=name)]
            if rng.random() < .5: exs.reverse()
            out.append(mk_pair("", kind, exs)); got += 1
    return out

def vqa_candidates(sp):
    Q = {q["question_id"]: q["question"] for q in json.load(open(vq(f"v2_OpenEnded_mscoco_{sp}2014_questions.json")))["questions"]}
    A = {a["question_id"]: a for a in json.load(open(vq(f"v2_mscoco_{sp}2014_annotations.json")))["annotations"]}
    P = json.load(open(vq(f"v2_mscoco_{sp}2014_complementary_pairs.json")))
    def info(qid):
        a = A[qid]; raws = [x["answer"] for x in a["answers"]]; norm = [normalize_answer(r) for r in raws]
        ans, c = Counter(norm).most_common(1)[0]
        if c < 8: return None
        orig = Counter(r for r, n in zip(raws, norm) if n == ans).most_common(1)[0][0]
        return dict(qid=qid, img=a["image_id"], q=Q[qid], a=orig, canon=ans, refs=raws, atype=a["answer_type"])
    cands = defaultdict(list)
    for q1, q2 in P:
        if q1 not in A or q2 not in A: continue
        i1 = info(q1); i2 = info(q2) if i1 else None
        if not i1 or not i2 or i1["canon"] == i2["canon"] or i1["atype"] != i2["atype"] or i1["img"] == i2["img"]: continue
        cands[i1["atype"]].append((i1, i2))
    return cands

def pick_vqa(cands, quota, used, rng):
    out = []
    for t, n in quota.items():
        pool = cands[t][:]; rng.shuffle(pool); got = 0
        for i1, i2 in pool:
            if i1["img"] in used or i2["img"] in used: continue
            used.update([i1["img"], i2["img"]])
            out.append(mk_pair("", t, [mk_ex(i["img"], i["q"], i["a"], "vqa", t, qid=i["qid"], refs=i["refs"]) for i in (i1, i2)]))
            got += 1
            if got == n: break
        assert got == n, f"not enough {t} pairs"
    return out

def build_data():
    rng = random.Random(SEED)
    caps = {sp: load_caps(sp) for sp in ("train", "val")}; inst = {sp: load_inst(sp) for sp in ("train", "val")}
    cv, ct = vqa_candidates("val"), vqa_candidates("train")
    D = {"dev": {}, "test": {}, "pool": {}}; used = set()
    D["test"]["vqa"]  = pick_vqa(cv, {"yes/no":36, "number":24, "other":36}, used, rng)
    D["dev"]["vqa"]   = pick_vqa(cv, {"yes/no":12, "number":8,  "other":12}, used, rng)
    cats, raw, clean = inst["val"]; vimgs = sorted(raw)
    D["test"]["ground"] = make_ground(cats, raw, clean, vimgs, {"presence":32, "counting":32}, rng, used)
    D["dev"]["ground"]  = make_ground(cats, raw, clean, vimgs, {"presence":12, "counting":12}, rng, used)
    ci = [i for i in sorted(caps["val"]) if i not in used]; rng.shuffle(ci); ci = ci[:96]; used.update(ci)
    D["cap_dev"] = [mk_ex(i, CAPQ, rng.choice(caps["val"][i]), "caption", "caption") for i in ci]
    ti = sorted(caps["train"]); rng.shuffle(ti)
    D["cap_train"] = [mk_ex(i, CAPQ, c, "caption", "caption") for i in ti[:6000] for c in rng.sample(caps["train"][i], 3)]
    cats, raw, clean = inst["train"]; gi = sorted(raw); rng.shuffle(gi); gi = gi[:3000]
    D["pool"]["presence"] = make_ground(cats, raw, clean, gi, {"presence":3000}, rng)
    D["pool"]["counting"] = make_ground(cats, raw, clean, gi, {"counting":3000}, rng)
    D["pool"]["vqa"] = pick_vqa(ct, {"yes/no":640, "number":256, "other":640}, set(), rng)
    for s in ("dev", "test"):
        for k in ("ground", "vqa"): tag(D[s][k], f"{s}-{k}"); assign_mismatch(D[s][k], SEED + 7)
    for k in D["pool"]: tag(D["pool"][k], f"train-{k}")
    ids = {e["img"] for e in D["cap_train"] + D["cap_dev"] + flat(sum(D["pool"].values(), [])) + flat(D["dev"]["ground"] + D["dev"]["vqa"] + D["test"]["ground"] + D["test"]["vqa"])}
    D["img_split"] = [[i, "train" if i in caps["train"] else "val"] for i in sorted(ids)]
    return D

def check_isolation(D):
    tr = {e["img"] for e in D["cap_train"]} | {e["img"] for k in D["pool"] for e in flat(D["pool"][k])}
    sets = {"cap_dev": {e["img"] for e in D["cap_dev"]}}
    for s in ("dev", "test"):
        for k in ("ground", "vqa"): sets[f"{s}_{k}"] = {e["img"] for e in flat(D[s][k])}
    names = list(sets)
    for i, a in enumerate(names):
        assert not (sets[a] & tr), f"{a} leaks into training/replay"
        for b in names[i+1:]: assert not (sets[a] & sets[b]), f"{a} overlaps {b}"
    assert len(D["cap_train"]) == 18000 and len(D["cap_dev"]) == 96
    assert len(D["pool"]["vqa"]) == 1536 and len(D["dev"]["vqa"]) == 32 and len(D["test"]["vqa"]) == 96
    assert len(D["dev"]["ground"]) == 24 and len(D["test"]["ground"]) == 64
    print("split isolation + sizes OK")

if os.path.exists(DATA_JSON): D = json.load(open(DATA_JSON))
else: D = build_data(); json.dump(D, open(DATA_JSON, "w"))
check_isolation(D)
IMG_SPLIT = {i: s for i, s in D["img_split"]}
print({k: len(v) for k, v in D["pool"].items()})

split isolation + sizes OK
{'presence': 3000, 'counting': 3000, 'vqa': 1536}


In [5]:
# ===================== CELL 3: SigLIP feature caching (timed separately) =====================
def cache_features():
    if os.path.exists(ROWS_JSON + ".stats"): return json.load(open(ROWS_JSON + ".stats"))
    from transformers import AutoImageProcessor, SiglipVisionModel
    ids = [i for i, _ in D["img_split"]]
    proc = AutoImageProcessor.from_pretrained(SIGLIP)
    enc = SiglipVisionModel.from_pretrained(SIGLIP, torch_dtype=torch.float16).to(dev).eval()
    class DS(torch.utils.data.Dataset):
        def __len__(s): return len(ids)
        def __getitem__(s, k):
            sp = IMG_SPLIT[ids[k]]; im = Image.open(f"{IMG_DIR[sp]}/COCO_{sp}2014_{ids[k]:012d}.jpg").convert("RGB")
            return proc(images=im, return_tensors="pt")["pixel_values"][0]
    dl = torch.utils.data.DataLoader(DS(), batch_size=128, num_workers=4)
    mm = np.lib.format.open_memmap(FEATS_NPY, mode="w+", dtype=np.float16, shape=(len(ids), 196, 768))
    torch.cuda.reset_peak_memory_stats(); t0 = time.time(); r = 0
    with torch.no_grad():
        for px in tqdm(dl, desc="caching"):
            h = enc(pixel_values=px.to(dev).half()).last_hidden_state     # post_layernorm already applied inside HF -> not applied again
            assert h.shape[1:] == (196, 768); mm[r:r+len(h)] = h.cpu().numpy(); r += len(h)
    mm.flush(); st = dict(seconds=time.time()-t0, peak_mem_gb=torch.cuda.max_memory_allocated()/2**30, n_images=len(ids))
    json.dump({i: k for k, i in enumerate(ids)}, open(ROWS_JSON, "w")); json.dump(st, open(ROWS_JSON + ".stats", "w"))
    del enc; torch.cuda.empty_cache(); print(st); return st
CACHE_STATS = cache_features()
FEATS = np.load(FEATS_NPY, mmap_mode="r"); ROW = {int(k): v for k, v in json.load(open(ROWS_JSON)).items()}
def get_feats(ids): return torch.from_numpy(np.stack([FEATS[ROW[i]] for i in ids])).to(dev)

preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  813MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

[transformers] SiglipVisionModel LOAD REPORT from: google/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp

caching:   0%|          | 0/92 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ===================== CELL 4: model, connector, batching, eval primitives =====================
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
tok = AutoTokenizer.from_pretrained(QWEN); IM_END = tok.convert_tokens_to_ids("<|im_end|>")
_t = tok.apply_chat_template([{"role":"user","content":"§IMG§"}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
PRE, SUF = _t.split("§IMG§")
assert PRE == "<|im_start|>user\n" and SUF.endswith("<think>\n\n</think>\n\n"), (PRE, SUF)
PRE_IDS, SUF_IDS = tok(PRE, add_special_tokens=False).input_ids, tok(SUF, add_special_tokens=False).input_ids
@lru_cache(None)
def tq(s): return tuple(tok(s, add_special_tokens=False).input_ids[:80])
@lru_cache(None)
def ta(s, cap): return tuple(tok(s, add_special_tokens=False).input_ids[:cap])
def AC(): return torch.autocast("cuda", dtype=torch.float16)

class Connector(nn.Module):
    def __init__(s, n_vis):
        super().__init__(); s.n_vis = n_vis
        s.mlp = nn.Sequential(nn.Linear(768, 1024), nn.GELU(), nn.Linear(1024, 1024))
        s.register_buffer("scale", torch.ones(()))
    def forward(s, f):                                    # f: [B,196,768], 14x14 row-major
        f = f.float(); B = f.shape[0]
        if s.n_vis == 49: f = f.view(B, 7, 2, 7, 2, 768).mean((2, 4)).reshape(B, 49, 768) if False else f.view(B, 14, 14, 768).view(B, 7, 2, 7, 2, 768).mean((2, 4)).reshape(B, 49, 768)
        return s.scale * s.mlp(f)

def init_model():
    global base, model, EMB, conn, opt, scaler, lora_params, S_RNG
    seed_all(SEED)
    base = AutoModelForCausalLM.from_pretrained(QWEN, torch_dtype=torch.float16, attn_implementation="sdpa").to(dev)
    base.config.use_cache = False
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    EMB = base.get_input_embeddings(); conn = Connector(N_VIS).to(dev)
    model = get_peft_model(base, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                           target_modules=["q_proj","k_proj","v_proj","o_proj"], task_type="CAUSAL_LM"))
    lora_params = [p for n, p in model.named_parameters() if "lora_" in n]
    for p in lora_params: p.data = p.data.float()
    assert all(("lora_" in n) == p.requires_grad for n, p in model.named_parameters()), "backbone must be frozen"
    opt = torch.optim.AdamW([{"params": list(conn.parameters()), "lr": 0.}, {"params": lora_params, "lr": 0.}], weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda"); S_RNG = np.random.default_rng(SEED + 2)

@torch.no_grad()
def calibrate(n=128):
    """s = E||e_text||_2 / E||g_phi(F)||_2 at init, measured on TRAINING examples only; Z = s * g_phi(F)."""
    r = random.Random(SEED); exs = r.sample(D["cap_train"], n - 32) + r.sample(flat(D["pool"]["vqa"]), 32)
    conn.scale.fill_(1.0); vis = conn(get_feats([e["img"] for e in exs])).norm(dim=-1).mean().item()
    ids = [t for e in exs for t in tq(e["q"]) + ta(e["a"], 64 if e["kind"] == "caption" else 16)]
    txt = EMB(torch.tensor(ids, device=dev)).float().norm(dim=-1).mean().item()
    conn.scale.fill_(txt / vis); c = dict(text_norm=txt, vis_norm_before=vis, scale=txt / vis); print("calibration:", c); return c

def build_batch(exs, cond="correct", include_end=True, with_target=True):
    key = {"correct":"img", "partner":"partner_img", "mismatch":"mm_img", "none":None}[cond]
    Z = conn(get_feats([e[key] for e in exs])).to(EMB.weight.dtype) if key else None
    pre = EMB(torch.tensor(PRE_IDS, device=dev)); seqs, labs, L = [], [], 1
    for b, e in enumerate(exs):
        q = list(tq(e["q"])); a = list(ta(e["a"], 64 if e["kind"] == "caption" else 16))
        tgt = (a + ([IM_END] if include_end else [])) if with_target else []
        seq = torch.cat([pre] + ([Z[b]] if key else []) + [EMB(torch.tensor(q + SUF_IDS + tgt, device=dev))], 0)
        lab = torch.full((seq.size(0),), -100, device=dev)
        if tgt: lab[-len(tgt):] = torch.tensor(tgt, device=dev)
        seqs.append(seq); labs.append(lab); L = max(L, len(tgt))
    B, T = len(exs), max(s.size(0) for s in seqs)
    emb = torch.zeros(B, T, seqs[0].size(1), dtype=seqs[0].dtype, device=dev)
    mask = torch.zeros(B, T, dtype=torch.long, device=dev); lab = torch.full((B, T), -100, device=dev)
    for b, (s, l) in enumerate(zip(seqs, labs)): n = s.size(0); emb[b, T-n:] = s; mask[b, T-n:] = 1; lab[b, T-n:] = l
    return dict(emb=emb, mask=mask, pos=(mask.cumsum(1) - 1).clamp(min=0), labels=lab, L=L)

def example_nll(B):
    out = model(inputs_embeds=B["emb"], attention_mask=B["mask"], position_ids=B["pos"], logits_to_keep=B["L"] + 1, use_cache=False)
    lg, lab = out.logits[:, :-1].float(), B["labels"][:, -B["L"]:]
    ce = Fn.cross_entropy(lg.reshape(-1, lg.size(-1)), lab.reshape(-1), ignore_index=-100, reduction="none").view(lab.shape)
    m = (lab != -100).float(); return (ce * m).sum(1) / m.sum(1).clamp(min=1)

@torch.no_grad()
def eval_nll(exs, cond, include_end, bs=8):
    model.eval(); out = []
    for i in range(0, len(exs), bs):
        with AC(): out.append(example_nll(build_batch(exs[i:i+bs], cond, include_end)).float().cpu())
    return torch.cat(out).numpy()

@torch.no_grad()
def generate(exs, cond, bs=8, max_new=16):
    model.eval(); preds, tt = [], 0.0
    for i in range(0, len(exs), bs):
        sub = exs[i:i+bs]; torch.cuda.synchronize(); t0 = time.time()
        with AC():
            B = build_batch(sub, cond, with_target=False); mask, last = B["mask"], B["pos"][:, -1]
            out = model(inputs_embeds=B["emb"], attention_mask=mask, position_ids=B["pos"], use_cache=True, logits_to_keep=1)
            past, nxt = out.past_key_values, out.logits[:, -1].argmax(-1)
            done = torch.zeros(len(sub), dtype=torch.bool, device=dev); toks = []
            for step in range(max_new):
                toks.append(torch.where(done, torch.full_like(nxt, IM_END), nxt)); done |= nxt == IM_END
                if done.all() or step == max_new - 1: break
                mask = torch.cat([mask, torch.ones_like(mask[:, :1])], 1); last = last + 1
                out = model(inputs_embeds=EMB(nxt)[:, None], attention_mask=mask, position_ids=last[:, None], past_key_values=past, use_cache=True)
                past, nxt = out.past_key_values, out.logits[:, -1].argmax(-1)
        torch.cuda.synchronize(); tt += time.time() - t0
        for row in torch.stack(toks, 1).tolist():
            row = row[:row.index(IM_END)] if IM_END in row else row; preds.append(tok.decode(row, skip_special_tokens=True).strip())
    return preds, tt

In [ ]:
# ===================== CELL 5: dev eval, sampler, schedule, checkpoints =====================
STAGES = {1: dict(n=2250, lr_c=1e-3, lr_l=0.), 2: dict(n=1500, lr_c=1e-4, lr_l=2e-5), 3: dict(n=750, lr_c=5e-5, lr_l=1e-5)}
if SMOKE:  STAGES[1]["n"], STAGES[2]["n"], STAGES[3]["n"] = 10, 6, 4        # tiny schedule: exercises all 3 stages
BOUNDS = [0, STAGES[1]["n"], STAGES[1]["n"] + STAGES[2]["n"], STAGES[1]["n"] + STAGES[2]["n"] + STAGES[3]["n"]]
TOTAL  = BOUNDS[-1]
EVAL_STEPS = (set(range(0, 4501, 500)) | {2250, 3750, 4500}) if not SMOKE else {0, 10, 16, 20}
CKPT_EVERY = 250 if not SMOKE else 5
MIX = {2: [("presence", .5), ("counting", .25), ("caption", .25)], 3: [("vqa", .7), ("presence", .1), ("counting", .1), ("caption", .1)]}
stage_of = lambda g: 1 if g < BOUNDS[1] else 2 if g < BOUNDS[2] else 3
STATS = {"trainable": {}, "stage_time_s": {"1":0., "2":0., "3":0.}, "peak_mem_gb": {"1":0., "2":0., "3":0.}, "mix": {"1":{}, "2":{}, "3":{}}}

def dev_eval(g):
    r = {"step": g, "cap_loss": float(eval_nll(D["cap_dev"], "correct", True).mean())}
    for s in ("ground", "vqa"):
        ex = flat(D["dev"][s]); c, p, n = (eval_nll(ex, k, False) for k in ("correct", "partner", "none"))
        r.update({f"{s}_nll_correct": float(c.mean()), f"{s}_nll_partner": float(p.mean()), f"{s}_nll_none": float(n.mean()),
                  f"{s}_dnll_partner": float((p - c).mean()), f"{s}_dnll_none": float((n - c).mean())})
    open(f"{OUT}/dev_log.jsonl", "a").write(json.dumps(r) + "\n"); print(r); return r

@lru_cache(2)
def _perm(ep): return np.random.default_rng(SEED + 100 + ep).permutation(len(D["cap_train"]))
def get_update_examples(st, g):
    if st == 1:
        N = len(D["cap_train"]); return [D["cap_train"][_perm(k // N)[k % N]] for k in range(g*16, g*16 + 16)]
    out = []
    for _ in range(8):
        r, acc = S_RNG.random(), 0.
        for k, p in MIX[st]:
            acc += p
            if r < acc: break
        if k == "caption": i, j = S_RNG.integers(len(D["cap_train"]), size=2); out += [D["cap_train"][i], D["cap_train"][j]]
        else: out += D["pool"][k][S_RNG.integers(len(D["pool"][k]))]["ex"]
    return out

def lr_factor(s, n):
    w = max(1, round(.05 * n)); return (s + 1) / w if s < w else .1 + .9 * .5 * (1 + math.cos(math.pi * (s - w) / max(1, n - w)))
def set_lr(st, local):
    f = lr_factor(local, STAGES[st]["n"]); opt.param_groups[0]["lr"] = STAGES[st]["lr_c"] * f; opt.param_groups[1]["lr"] = STAGES[st]["lr_l"] * f
def setup_stage(st):
    for p in conn.parameters(): p.requires_grad_(True)
    for p in lora_params: p.requires_grad_(st >= 2)
    tr = [p for p in list(conn.parameters()) + lora_params if p.requires_grad]
    STATS["trainable"][str(st)] = sum(p.numel() for p in tr); torch.cuda.reset_peak_memory_stats(); return tr
lora_sd = lambda: {n: p.detach().cpu() for n, p in model.named_parameters() if "lora_" in n}

def save_ckpt(g):
    tmp = f"{OUT}/ckpt_latest.pt.tmp"
    torch.save(dict(g=g, conn=conn.state_dict(), lora=lora_sd(), opt=opt.state_dict(), scaler=scaler.state_dict(), s_rng=S_RNG.bit_generator.state,
                    torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state(), stats=STATS), tmp); os.replace(tmp, f"{OUT}/ckpt_latest.pt")
def try_resume():
    p = f"{OUT}/ckpt_latest.pt"
    if not os.path.exists(p): return 0
    ck = torch.load(p, map_location="cpu", weights_only=False)
    conn.load_state_dict(ck["conn"]); model.load_state_dict(ck["lora"], strict=False); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
    S_RNG.bit_generator.state = ck["s_rng"]; torch.set_rng_state(ck["torch_rng"]); torch.cuda.set_rng_state(ck["cuda_rng"].cpu())
    STATS.update(ck["stats"]); print("resumed at", ck["g"]); return ck["g"]

In [ ]:
# ===================== CELL 6: sanity checks + training =====================
def sanity_checks():
    f = get_feats([D["cap_train"][0]["img"]]); assert f.shape == (1, 196, 768) and f.dtype == torch.float16
    tr = setup_stage(1); assert len(tr) == len(list(conn.parameters())) and all(p.requires_grad for p in conn.parameters())
    assert all(not p.requires_grad for n, p in model.named_parameters())
    model.train(); exs = D["cap_train"][:8]
    with AC(): B = build_batch(exs); loss = example_nll(B).mean()
    print("supervised text ex0:", repr(tok.decode(B["labels"][0][B["labels"][0] != -100])), "| answer:", repr(exs[0]["a"]))
    loss.backward(); gn = sum(p.grad.float().norm()**2 for p in conn.parameters() if p.grad is not None) ** .5
    assert gn > 0, "no gradient reaches connector"; assert all(p.grad is None for n, p in model.named_parameters())
    opt.zero_grad(set_to_none=True); print(f"connector grad norm {gn:.3e} | trainables stage1: {STATS['trainable']['1']:,} | checks OK")

def train():
    g = try_resume(); cur = None; logf = open(f"{OUT}/train_log.jsonl", "a")
    if g == 0: dev_eval(0)
    while g < TOTAL:
        st = stage_of(g)
        if st != cur: tr = setup_stage(st); cur = st
        set_lr(st, g - BOUNDS[st - 1]); t0 = time.time()
        exs = get_update_examples(st, g); opt.zero_grad(set_to_none=True); model.train(); tot = 0.
        for m in range(2):                                                     # microbatch 8, accumulation 2
            with AC(): loss = example_nll(build_batch(exs[m*8:(m+1)*8], "correct", True)).mean() / 2
            scaler.scale(loss).backward(); tot += loss.item()
        scaler.unscale_(opt); gn = torch.nn.utils.clip_grad_norm_(tr, 1.0).item()
        scaler.step(opt); scaler.update(); torch.cuda.synchronize(); g += 1
        mix = Counter(e["kind"] for e in exs)
        for k, v in mix.items(): STATS["mix"][str(st)][k] = STATS["mix"][str(st)].get(k, 0) + v
        STATS["stage_time_s"][str(st)] += time.time() - t0
        STATS["peak_mem_gb"][str(st)] = max(STATS["peak_mem_gb"][str(st)], torch.cuda.max_memory_allocated() / 2**30)
        logf.write(json.dumps(dict(step=g, stage=st, loss=tot, gnorm=gn, lr_c=opt.param_groups[0]["lr"], lr_l=opt.param_groups[1]["lr"],
                                   amp_scale=scaler.get_scale(), mix=dict(mix))) + "\n")
        if g % 50 == 0 or SMOKE: logf.flush(); print(f"step {g}/{TOTAL} stage {st} loss {tot:.4f} gn {gn:.2f}")
        if g in EVAL_STEPS: dev_eval(g)
        if g % CKPT_EVERY == 0: save_ckpt(g)
        if g in BOUNDS[1:]: torch.save(dict(conn=conn.state_dict(), lora=lora_sd()), f"{OUT}/stage{stage_of(g-1)}.pt")
    json.dump(dict(STATS=STATS, cache=CACHE_STATS, calib=CALIB, n_vis=N_VIS, stages=STAGES, seed=SEED), open(f"{OUT}/stats.json", "w"), indent=1)

init_model(); CALIB = calibrate(); sanity_checks()
train()

In [ ]:
# ===================== CELL 7: final test evaluation (run ONCE after training) =====================
CONDS = ["correct", "partner", "mismatch", "none"]
def eval_suite(suite):
    pairs = D["test"][suite]; exs = flat(pairs); R = {}
    for c in CONDS:
        preds, t = generate(exs, c); nll = eval_nll(exs, c, False)
        pn = [normalize_answer(p) for p in preds]; exact = np.array([a == e["canon"] for a, e in zip(pn, exs)], float)
        acc = exact if suite == "ground" else np.array([vqa_soft(a, e["refs"]) for a, e in zip(pn, exs)])
        R[c] = dict(acc=acc, exact=exact, empty=np.array([a == "" for a in pn], float), nll=nll, pn=np.array(pn), ms=1000 * t / len(exs))
        json.dump([dict(pid=e["pid"], q=e["q"], img=e["img"], refs=e["refs"], canon=e["canon"], pred=p, pred_norm=a, acc=float(x), nll=float(n))
                   for e, p, a, x, n in zip(exs, preds, pn, acc, nll)], open(f"{OUT}/test_{suite}_{c}.json", "w"), indent=1)
    return R

def report(suite, R, B=4000):
    pairs = D["test"][suite]; n = len(pairs); typ = np.array([p["type"] for p in pairs]); rng = np.random.default_rng(SEED)
    canon = np.array([e["canon"] for e in flat(pairs)]).reshape(n, 2)
    def pl(c):
        r = R[c]; two = lambda x: x.reshape(n, 2)
        d = dict(acc=two(r["acc"]).mean(1), pair_acc=two(r["exact"]).all(1).astype(float), identical=(two(r["pn"])[:, 0] == two(r["pn"])[:, 1]).astype(float),
                 empty=two(r["empty"]).mean(1), nll=two(r["nll"]).mean(1))
        pn = two(r["pn"]); d["follows_partner"] = (pn == canon[:, ::-1]).mean(1); d["stays_original"] = (pn == canon).mean(1)
        d["other"] = 1 - d["follows_partner"] - d["stays_original"]; return d
    M = {c: pl(c) for c in CONDS}; rows = []
    for name, sub in [("all", np.arange(n))] + [(t, np.where(typ == t)[0]) for t in sorted(set(typ))]:
        idx = rng.integers(0, len(sub), (B, len(sub)))
        ci = lambda v: np.percentile(v[sub][idx].mean(1), [2.5, 97.5])
        for c in CONDS:
            for k in ["acc", "pair_acc", "identical", "empty", "nll"] + (["follows_partner", "stays_original", "other"] if c == "partner" else []):
                lo, hi = ci(M[c][k]); rows.append(dict(suite=suite, subset=name, cond=c, metric=k, mean=M[c][k][sub].mean(), lo=lo, hi=hi, n_pairs=len(sub)))
        for c in CONDS[1:]:
            for k, sgn in (("acc", 1), ("pair_acc", 1), ("nll", -1)):
                v = sgn * (M["correct"][k] - M[c][k]); lo, hi = ci(v); rows.append(dict(suite=suite, subset=name, cond=f"adv_vs_{c}", metric=k, mean=v[sub].mean(), lo=lo, hi=hi, n_pairs=len(sub)))
    for c in CONDS: rows.append(dict(suite=suite, subset="all", cond=c, metric="gen_ms_per_answer", mean=R[c]["ms"], lo=np.nan, hi=np.nan, n_pairs=n))
    return pd.DataFrame(rows), M

def qualitative(suite, R, k=3):
    """Selection rule (seeded): k random pairs per bucket. grounded = correct on both originals AND both swap answers follow the partner image;
       language-prior = swapped predictions identical to original predictions for both; other-change = swap answers match neither original nor partner."""
    pairs = D["test"][suite]; n = len(pairs); r = random.Random(SEED); pn = {c: R[c]["pn"].reshape(n, 2) for c in CONDS}
    canon = np.array([e["canon"] for e in flat(pairs)]).reshape(n, 2); ok = lambda c: (pn[c] == canon).all(1)
    sw_follow = (pn["partner"] == canon[:, ::-1]).all(1); sw_same = (pn["partner"] == pn["correct"]).all(1)
    other = (pn["partner"] != canon).all(1) & (pn["partner"] != canon[:, ::-1]).all(1)
    for name, m in [("grounded", ok("correct") & sw_follow), ("language-prior", sw_same), ("other-change", other)]:
        for i in r.sample(list(np.where(m)[0]), min(k, int(m.sum()))):
            p = pairs[i]; print(f"[{suite}/{name}] {p['pid']} Q: {p['ex'][0]['q']}")
            for j, e in enumerate(p["ex"]): print(f"   img {e['img']} refs={e['canon']!r} | correct={pn['correct'][i][j]!r} swap={pn['partner'][i][j]!r} mismatch={pn['mismatch'][i][j]!r} none={pn['none'][i][j]!r}")

ALL, RES = [], {}
torch.cuda.reset_peak_memory_stats(); t_eval = time.time()
for s in ("ground", "vqa"):
    RES[s] = eval_suite(s); df, M = report(s, RES[s]); ALL.append(df)
    print(df[df.subset == "all"].pivot_table(index="cond", columns="metric", values="mean").round(3))
EVAL_INFO = dict(eval_seconds=time.time() - t_eval, eval_peak_mem_gb=torch.cuda.max_memory_allocated() / 2**30)
pd.concat(ALL).to_csv(f"{OUT}/test_results.csv", index=False)

In [ ]:
# ===================== CELL 8: learning curves =====================
def plot_curves():
    import matplotlib.pyplot as plt
    tl = pd.read_json(f"{OUT}/train_log.jsonl", lines=True); dl = pd.read_json(f"{OUT}/dev_log.jsonl", lines=True)
    fig, ax = plt.subplots(2, 3, figsize=(16, 8)); ax = ax.ravel()
    ax[0].plot(tl.step, tl.loss.rolling(50, min_periods=1).mean()); ax[0].set_title("train loss (50-step mean)")
    ax[1].plot(dl.step, dl.cap_loss, "o-"); ax[1].set_title("dev caption loss")
    for s in ("ground", "vqa"):
        ax[2].plot(dl.step, dl[f"{s}_nll_correct"], "o-", label=s); ax[3].plot(dl.step, dl[f"{s}_dnll_partner"], "o-", label=s); ax[4].plot(dl.step, dl[f"{s}_dnll_none"], "o-", label=s)
    ax[2].set_title("dev answer NLL (correct image)"); ax[3].set_title(r"$\Delta$NLL (partner $-$ correct)"); ax[4].set_title(r"$\Delta$NLL (no image $-$ correct)")
    for a in ax[:5]:
        for b in BOUNDS[1:3]: a.axvline(b, ls="--", c="gray")
    ax[2].legend(); ax[5].axis("off"); plt.tight_layout(); plt.savefig(f"{OUT}/curves.png", dpi=150); plt.show()
plot_curves()

In [ ]:
# ===================== CELL 9: save everything the report needs =====================
import io, contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    for s in ("ground", "vqa"): qualitative(s, RES[s], k=4)
open(f"{OUT}/qualitative_examples.txt", "w").write(buf.getvalue()); print(buf.getvalue())

cfg = dict(seed=SEED, n_vis=N_VIS, smoke=SMOKE, models=dict(llm=QWEN, vision=SIGLIP), stages=STAGES, bounds=BOUNDS,
           microbatch=8, accum=2, effective_batch=16, optimizer="AdamW", weight_decay=0.01, clip=1.0, warmup_frac=0.05, final_lr_frac=0.1,
           lora=dict(r=16, alpha=32, dropout=0.05, targets=["q_proj","k_proj","v_proj","o_proj"]), mix=MIX,
           caps=dict(question=80, caption=64, answer=16), gen=dict(max_new=16, greedy=True, bs=8), bootstrap=4000, min_area=MIN_AREA)
json.dump(dict(config=cfg, eval=EVAL_INFO, stats=STATS, cache=CACHE_STATS, calib=CALIB), open(f"{OUT}/config_and_resources.json", "w"), indent=1)

res = pd.concat(ALL); L = []
L.append("== RESOURCES ==")
L.append(f"feature caching: {CACHE_STATS}")
for st in "123": L.append(f"stage {st}: trainable={STATS['trainable'][st]:,}  time={STATS['stage_time_s'][st]:.0f}s  peak_mem={STATS['peak_mem_gb'][st]:.2f}GB  mix={STATS['mix'][st]}")
L.append(f"test eval: {EVAL_INFO}")
L.append("\n== DEV (last rows) ==\n" + pd.read_json(f"{OUT}/dev_log.jsonl", lines=True).round(4).to_string(index=False))
for s in ("ground", "vqa"):
    d = res[res.suite == s]
    L.append(f"\n== TEST {s}: means by condition/subset ==")
    L.append(d[~d.cond.str.startswith("adv")].pivot_table(index=["subset","cond"], columns="metric", values="mean").round(3).to_string())
    L.append(f"\n== TEST {s}: correct-image advantages (mean [95% CI]) ==")
    a = d[d.cond.str.startswith("adv") & d.metric.isin(["acc","pair_acc","nll"])].copy()
    a["val"] = a.apply(lambda r: f"{r['mean']:.3f} [{r['lo']:.3f}, {r['hi']:.3f}]", axis=1)
    L.append(a.pivot_table(index=["subset","cond"], columns="metric", values="val", aggfunc="first").to_string())
    f = d[(d.cond == "partner") & d.metric.isin(["follows_partner","stays_original","other"])].pivot_table(index="subset", columns="metric", values="mean").round(3)
    L.append(f"\n== TEST {s}: under complementary swap, what the answer does ==\n" + f.to_string())
open(f"{OUT}/summary.txt", "w").write("\n".join(L)); print("\n".join(L))
import shutil; shutil.make_archive(f"{WORK}/results_{os.path.basename(OUT)}", "zip", OUT)   # download this zip